In [35]:
import os

src_root = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/iclr_2026_processed_data/final_data/pak_punjab/val"
dst_root = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/pretraing"
dst_flat = os.path.join(dst_root, "low_resolution")  # single common folder

os.makedirs(dst_flat, exist_ok=True)

# collect all directories literally named "rgb" (case-insensitive match)
rgb_dirs = []
for root, dirs, _ in os.walk(src_root):
    for d in dirs:
        if d.lower() == "images":
            rgb_dirs.append(os.path.join(root, d))

print(f"Found {len(rgb_dirs)} 'images' folders")

def unique_path(base_dir: str, filename: str) -> str:
    """Ensure unique filename inside base_dir by appending _1, _2, ... if needed."""
    name, ext = os.path.splitext(filename)
    candidate = os.path.join(base_dir, filename)
    k = 1
    while os.path.exists(candidate):
        candidate = os.path.join(base_dir, f"{name}_{k}{ext}")
        k += 1
    return candidate

link_count = 0
skip_count = 0

for rgb_dir in rgb_dirs:
    # Parent folder name used as prefix to reduce collisions across states/regions
    parent = os.path.basename(os.path.dirname(rgb_dir))

    for fname in os.listdir(rgb_dir):
        if not fname.lower().endswith(".png"):
            continue
        src_path = os.path.join(rgb_dir, fname)

        # prefix with parent to avoid most collisions; still ensure uniqueness
        prefixed_name = f"{fname}"
        dst_link = unique_path(dst_flat, prefixed_name)

        try:
            os.symlink(src_path, dst_link)
            link_count += 1
        except FileExistsError:
            # extremely unlikely due to unique_path; keep for safety
            skip_count += 1
        except OSError as e:
            # e.g., filesystem without symlink support or permission issues
            print(f"Skip {src_path}: {e}")
            skip_count += 1

print(f"Symlinking complete. Linked .png files: {link_count}. Skipped: {skip_count}.")
print(f"Destination: {dst_flat}")

Found 1 'images' folders
Symlinking complete. Linked .png files: 2873. Skipped: 0.
Destination: /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/pretraing/low_resolution


In [ ]:
# import os
# import shutil

# pretrain_root = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/pretraing"
# keep_folder = "high_resolution"

# for item in os.listdir(pretrain_root):
#     path = os.path.join(pretrain_root, item)
#     if item == keep_folder:
#         continue
#     try:
#         if os.path.isdir(path):
#             shutil.rmtree(path)
#         else:
#             os.remove(path)
#     except Exception as e:
#         print(f"Error deleting {path}: {e}")

# print(f"Cleanup complete. Only '{keep_folder}' folder is kept in {pretrain_root}.")

Cleanup complete. Only 'high_resolution' folder is kept in /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/pretraing.


In [27]:
import os

dst_root = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/pretraing/low_resolution"

total_files = 0
symlinked = 0
broken_links = 0
valid_links = 0

for fname in os.listdir(dst_root):
    path = os.path.join(dst_root, fname)
    if os.path.islink(path):
        symlinked += 1
        target = os.readlink(path)
        if os.path.exists(target):
            valid_links += 1
        else:
            broken_links += 1
    total_files += 1

print(f"Total entries: {total_files}")
print(f"Symlinks: {symlinked}")
print(f"Valid symlinks: {valid_links}")
print(f"Broken symlinks: {broken_links}")

if total_files == symlinked and broken_links == 0:
    print("Verification passed: all images are correctly symlinked.")
else:
    print("Verification failed: some files are not symlinks or are broken.")

FileNotFoundError: [Errno 2] No such file or directory: '/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/pretraing/low_resolution'

In [36]:
import os
from collections import Counter

dst_root = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/pretraing/low_resolution"

# collect all filenames (without symlink path)
filenames = [f for f in os.listdir(dst_root) if os.path.islink(os.path.join(dst_root, f))]

counts = Counter(filenames)
duplicates = [f for f, c in counts.items() if c > 1]

print(f"Total images: {len(filenames)}")
print(f"Unique images: {len(counts)}")

if duplicates:
    print(f"Duplicate file names found: {len(duplicates)}")
    for d in duplicates[:20]:
        print(f"  {d}")
else:
    print("No duplicate filenames found in low_resotion folder.")
    

Total images: 31895
Unique images: 31895
No duplicate filenames found in low_resotion folder.


In [4]:
import os

dst_root = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/pretraing/low_resotion"

total_symlink_size = 0
unique_targets = set()

for fname in os.listdir(dst_root):
    path = os.path.join(dst_root, fname)
    if os.path.islink(path):
        target = os.readlink(path)
        # resolve relative symlinks
        if not os.path.isabs(target):
            target = os.path.join(os.path.dirname(path), target)
        if os.path.exists(target):
            # count actual file only once (unique physical file)
            if target not in unique_targets:
                total_symlink_size += os.path.getsize(target)
                unique_targets.add(target)

# Convert to human-readable size
def format_size(bytes_val):
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if bytes_val < 1024:
            return f"{bytes_val:.2f} {unit}"
        bytes_val /= 1024
    return f"{bytes_val:.2f} PB"

print(f"Number of unique linked files: {len(unique_targets)}")
print(f"Total size of linked image data: {format_size(total_symlink_size)}")

Number of unique linked files: 2547708
Total size of linked image data: 75.41 GB


In [20]:
import dataset_tools as dtools
dtools.download(dataset='xView 2018', dst_dir='~/dataset-ninja/')


'/home/rishabh.mondal/dataset-ninja/xview-2018.tar'

In [21]:
import tarfile

tar_path = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/iclr_2026_processed_data/xview-2018.tar"
extract_path = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/iclr_2026_processed_data/"

with tarfile.open(tar_path, "r") as tar:
    tar.extractall(path=extract_path)

print("Extraction complete.")

ReadError: file could not be opened successfully:
- method gz: ReadError('not a gzip file')
- method bz2: ReadError('not a bzip2 file')
- method xz: ReadError('not an lzma file')
- method tar: ReadError('invalid header')

In [37]:
import os
from PIL import Image

src_dir = "/home/kirtangangani/satellite_data/xVIEW_dataset/1/train_images/train_images"
# dst_dir = os.path.join(src_dir, "converted_png")
dst_dir="/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/pretraing/high_resolution"
# os.makedirs(dst_dir, exist_ok=True)

count = 0
for root, _, files in os.walk(src_dir):
    for file in files:
        if file.lower().endswith(".tif") or file.lower().endswith(".tiff"):
            src_path = os.path.join(root, file)
            dst_path = os.path.join(dst_dir, os.path.splitext(file)[0] + ".png")

            try:
                with Image.open(src_path) as img:
                    img.convert("RGB").save(dst_path, "PNG", compress_level=1)
                count += 1
            except Exception as e:
                print(f"Failed to convert {src_path}: {e}")

print(f"Conversion complete. {count} images converted to {dst_dir}")

Conversion complete. 846 images converted to /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/pretraing/high_resolution
